# Multi-Tissue Comparison Analysis

Compare multiple tissue samples:
- Integrate multiple Xenium datasets
- Batch correction
- Cross-tissue cell type comparison
- Tissue-specific spatial patterns

**Input:** Multiple preprocessed and annotated AnnData objects

**Output:** Integrated multi-tissue dataset with comparative analysis

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sc.settings.set_figure_params(dpi=80)
print(f"Scanpy version: {sc.__version__}")
print(f"scVI version: {scvi.__version__}")

## 1. Load Multiple Tissue Samples

In [ ]:
# Define paths
DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../data/processed")
FIGURES_DIR = Path("../figures/05_tissue_comparisons")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# List of sample names to integrate
sample_names = [
    "xenium_sample_01",
    # "xenium_sample_02",
    # "xenium_sample_03",
    # Add more samples
]

# Load samples
adatas = []
for sample in sample_names:
    try:
        adata = sc.read_h5ad(DATA_DIR / f"{sample}_annotated.h5ad")
        adata.obs['sample_id'] = sample
        adatas.append(adata)
        print(f"Loaded {sample}: {adata.shape}")
    except FileNotFoundError:
        print(f"Warning: {sample} not found, skipping")

if len(adatas) == 0:
    print("\nNo samples loaded. Please ensure sample files exist.")
    print("For demonstration, loading single sample...")
    adata = sc.read_h5ad(DATA_DIR / "xenium_sample_01_annotated.h5ad")
    adata.obs['sample_id'] = 'sample_01'
else:
    print(f"\nTotal samples loaded: {len(adatas)}")

## 2. Concatenate Samples

In [ ]:
# Concatenate all samples if multiple exist
if len(adatas) > 1:
    adata_combined = adatas[0].concatenate(
        adatas[1:],
        batch_key='sample_batch',
        batch_categories=[f'batch_{i}' for i in range(len(adatas))],
        index_unique='-'
    )
    print(f"\nCombined dataset shape: {adata_combined.shape}")
else:
    adata_combined = adata.copy()
    adata_combined.obs['sample_batch'] = 'batch_0'
    print(f"\nSingle sample mode: {adata_combined.shape}")

print(f"Samples: {adata_combined.obs['sample_id'].nunique()}")
print(f"Total cells: {adata_combined.n_obs}")

## 3. Pre-Integration QC

In [ ]:
# Plot cell type composition per sample
composition = pd.crosstab(
    adata_combined.obs['sample_id'],
    adata_combined.obs['celltype'],
    normalize='index'
) * 100

print("\nCell type composition (%) by sample:")
print(composition)

# Plot
composition.T.plot(kind='bar', figsize=(12, 6), stacked=False)
plt.ylabel('Percentage')
plt.xlabel('Cell Type')
plt.title('Cell Type Composition Across Samples')
plt.legend(title='Sample', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'composition_by_sample.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Batch Correction with scVI

In [ ]:
if len(adatas) > 1:
    # Setup scVI for batch correction
    scvi.model.SCVI.setup_anndata(
        adata_combined,
        layer='counts',
        batch_key='sample_batch'
    )
    
    # Train scVI model
    vae = scvi.model.SCVI(
        adata_combined,
        n_layers=2,
        n_latent=30,
        gene_likelihood="nb"
    )
    
    print("Training scVI for batch correction...")
    vae.train(max_epochs=200, early_stopping=True)
    
    # Get batch-corrected latent representation
    adata_combined.obsm['X_scvi'] = vae.get_latent_representation()
    
    # Compute integrated embedding
    sc.pp.neighbors(adata_combined, use_rep='X_scvi')
    sc.tl.umap(adata_combined)
    sc.tl.leiden(adata_combined, resolution=0.5, key_added='leiden_integrated')
    
    print("Batch correction completed")
else:
    print("Single sample - batch correction not needed")
    # Use existing UMAP if available
    if 'X_umap' not in adata_combined.obsm:
        sc.pp.neighbors(adata_combined)
        sc.tl.umap(adata_combined)

## 5. Visualize Integration

In [ ]:
# Plot UMAP colored by sample and cell type
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

sc.pl.umap(adata_combined, color='sample_id', ax=axes[0, 0], show=False, title='Sample ID')
sc.pl.umap(adata_combined, color='celltype', ax=axes[0, 1], show=False, title='Cell Type')
sc.pl.umap(adata_combined, color='total_counts', ax=axes[1, 0], show=False, cmap='viridis', title='Total Counts')
sc.pl.umap(adata_combined, color='n_genes_by_counts', ax=axes[1, 1], show=False, cmap='viridis', title='N Genes')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'integrated_umap.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Cross-Tissue DE Analysis

In [ ]:
# Find genes differentially expressed across samples
if adata_combined.obs['sample_id'].nunique() > 1:
    # Create log_normalized layer if needed
    if 'log_normalized' not in adata_combined.layers:
        adata_combined.X = adata_combined.layers['counts'].copy()
        sc.pp.normalize_total(adata_combined)
        sc.pp.log1p(adata_combined)
        adata_combined.layers['log_normalized'] = adata_combined.X.copy()
        adata_combined.X = adata_combined.layers['scaled'].copy()

    # Subsample for DE (1000 cells per group)
    n_per_group = 1000
    np.random.seed(42)
    indices = []
    for group in adata_combined.obs['sample_id'].unique():
        group_idx = adata_combined.obs.index[adata_combined.obs['sample_id'] == group]
        indices.extend(np.random.choice(group_idx, size=min(n_per_group, len(group_idx)), replace=False))
    adata_de = adata_combined[indices].copy()

    sc.tl.rank_genes_groups(
        adata_de,
        groupby='sample_id',
        method='wilcoxon',
        layer='log_normalized',
        key_added='de_samples',
        use_raw=False
    )

    # Copy results back to main object
    adata_combined.uns['de_samples'] = adata_de.uns['de_samples']

    # Plot
    sc.pl.rank_genes_groups(adata_combined, n_genes=20, sharey=False, key='de_samples')
    plt.savefig(FIGURES_DIR / 'de_genes_across_samples.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Save results
    de_results = sc.get.rank_genes_groups_df(adata_combined, group=None, key='de_samples')
    de_results.to_csv(OUTPUT_DIR / 'tissue_comparison_de_genes.csv', index=False)
    print(f"\nDE results saved: {len(de_results)} entries")
else:
    print("Single sample - cross-tissue DE not applicable")

## 7. Cell Type-Specific Tissue Differences

In [ ]:
# Analyze tissue differences for each cell type
if adata_combined.obs['sample_id'].nunique() > 1:
    # Ensure log_normalized layer exists
    if 'log_normalized' not in adata_combined.layers:
        adata_combined.X = adata_combined.layers['counts'].copy()
        sc.pp.normalize_total(adata_combined)
        sc.pp.log1p(adata_combined)
        adata_combined.layers['log_normalized'] = adata_combined.X.copy()
        adata_combined.X = adata_combined.layers['scaled'].copy()

    celltype_tissue_de = []

    for ct in adata_combined.obs['celltype'].unique():
        print(f"\nAnalyzing {ct}...")

        # Subset
        adata_ct = adata_combined[adata_combined.obs['celltype'] == ct].copy()

        # Check if we have multiple samples
        if adata_ct.obs['sample_id'].nunique() < 2 or adata_ct.n_obs < 20:
            print(f"  Skipping - insufficient data")
            continue

        # Subsample for DE (1000 cells per group)
        n_per_group = 1000
        np.random.seed(42)
        indices = []
        for group in adata_ct.obs['sample_id'].unique():
            group_idx = adata_ct.obs.index[adata_ct.obs['sample_id'] == group]
            indices.extend(np.random.choice(group_idx, size=min(n_per_group, len(group_idx)), replace=False))
        adata_de = adata_ct[indices].copy()

        # Run DE on subsampled data
        sc.tl.rank_genes_groups(
            adata_de,
            groupby='sample_id',
            method='wilcoxon',
            layer='log_normalized',
            use_raw=False
        )

        # Copy results back
        adata_ct.uns['rank_genes_groups'] = adata_de.uns['rank_genes_groups']

        # Get results
        ct_de = sc.get.rank_genes_groups_df(adata_ct, group=None)
        ct_de['celltype'] = ct
        celltype_tissue_de.append(ct_de)
        print(f"  {len(ct_de)} DE genes found")

    # Combine
    if celltype_tissue_de:
        all_ct_de = pd.concat(celltype_tissue_de, ignore_index=True)
        all_ct_de.to_csv(
            OUTPUT_DIR / 'celltype_tissue_de_genes.csv',
            index=False
        )
        print(f"\nCombined results: {len(all_ct_de)} entries")

## 8. Tissue-Specific Spatial Patterns (if applicable)

In [ ]:
# Compare spatial patterns across tissues
# This requires spatial coordinates to be preserved during concatenation

if 'spatial' in adata_combined.obsm and len(adatas) > 1:
    import squidpy as sq
    
    # Calculate neighborhood enrichment per sample
    for sample in adata_combined.obs['sample_id'].unique():
        print(f"\nAnalyzing spatial patterns in {sample}...")
        
        adata_sample = adata_combined[adata_combined.obs['sample_id'] == sample].copy()
        
        # Build spatial graph
        sq.gr.spatial_neighbors(adata_sample, coord_type='generic', n_neighs=10)
        
        # Neighborhood enrichment
        sq.gr.nhood_enrichment(adata_sample, cluster_key='celltype')
        
        # Plot
        sq.pl.nhood_enrichment(
            adata_sample,
            cluster_key='celltype',
            title=f'{sample} Neighborhood Enrichment'
        )
        plt.savefig(FIGURES_DIR / f'nhood_enrichment_{sample}.png', dpi=300, bbox_inches='tight')
        plt.show()
else:
    print("Spatial coordinates not available for comparison")

## 9. Save Integrated Data

In [ ]:
# Save integrated dataset
output_file = OUTPUT_DIR / 'integrated_tissues.h5ad'
adata_combined.write_h5ad(output_file)

print(f"\nIntegrated data saved to: {output_file}")
print(f"Total cells: {adata_combined.n_obs}")
print(f"Total samples: {adata_combined.obs['sample_id'].nunique()}")
print(f"Cell types: {adata_combined.obs['celltype'].nunique()}")